# 📈 Notebook 03 — Univariate EDA

Distributions, skewness, kurtosis, and domain interpretation for all features.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plt.rcParams['figure.facecolor'] = '#0a0f1e'
plt.rcParams['axes.facecolor'] = '#1e293b'
plt.rcParams['text.color'] = 'white'
sns.set_theme(style='darkgrid')

RAW = Path('../data/raw/students.csv')
PROCESSED = Path('../data/processed/students_processed.csv')
df_raw = pd.read_csv(RAW) if RAW.exists() else None
df = pd.read_csv(PROCESSED) if PROCESSED.exists() else df_raw
print(f'Loaded: {len(df):,} rows × {df.shape[1]} columns')


In [ ]:
numeric_cols = ['attendance_percentage', 'avg_assignment_score',
                'lms_login_frequency', 'library_visits_per_month',
                'disciplinary_actions', 'engagement_index',
                'academic_risk_score', 'composite_dropout_risk']

fig, axes = plt.subplots(3, 3, figsize=(16, 12), facecolor='#0a0f1e')
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=40, color='#38bdf8', edgecolor='#0a0f1e', alpha=0.85)
    axes[i].set_title(col, color='white', fontsize=11)
    axes[i].tick_params(colors='#94a3b8')
    axes[i].set_facecolor('#1e293b')
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Univariate Distributions', color='white', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import skew, kurtosis

skew_kurt = pd.DataFrame({
    'skewness': df[numeric_cols].apply(lambda x: round(skew(x.dropna()), 4)),
    'kurtosis': df[numeric_cols].apply(lambda x: round(kurtosis(x.dropna()), 4)),
    'mean': df[numeric_cols].mean().round(3),
    'std': df[numeric_cols].std().round(3),
})
skew_kurt


In [ ]:
# Categorical distributions
cat_cols = ['semester', 'hostel_resident', 'internet_access', 'dropout']
fig = make_subplots(rows=1, cols=4, subplot_titles=cat_cols)
for i, col in enumerate(cat_cols, 1):
    vc = df[col].value_counts()
    fig.add_trace(go.Bar(x=vc.index.astype(str), y=vc.values,
                         marker_color='#818cf8', name=col), row=1, col=i)
fig.update_layout(template='plotly_dark', height=350, showlegend=False,
                  title_text='Categorical Distributions')
fig.show()


## 💡 Key Takeaways

- **Technical:** `attendance_percentage` is left-skewed (many high-attenders). `disciplinary_actions` is heavily right-skewed (most students have 0). `composite_dropout_risk` shows bimodal separation.

- **Business:** The bimodal risk distribution means the student population naturally divides into 'safe' and 'at-risk' groups — interventions can be cleanly targeted rather than applied campus-wide.

- **Recommendation:** Use the natural 0.5 threshold in `composite_dropout_risk` as the operational cut-off for intervention triggers.